In [7]:
from pathlib import Path

import pandas as pd
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))
import matplotlib.pyplot as plt

TensorFlow version: 2.21.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

class_names = [
    "Glioma",
    "Meningioma",
    "No Tumor",
    "Pituitary"
]

In [9]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))

print(train_df.head())

Train: 4263
Validation: 512
   split       class                                              image  \
0  train      Glioma     ../data/raw/train/Glioma/images/Tr-gl_1184.jpg   
1  train  Meningioma  ../data/raw/train/Meningioma/images/Tr-me_0190...   
2  train   Pituitary     ../data/raw/train/Pituitary/images/p (669).jpg   
3  train  Meningioma  ../data/raw/train/Meningioma/images/Tr-me_0501...   
4  train   Pituitary  ../data/raw/train/Pituitary/images/Tr-pi_0777.jpg   

                                          annotation  label  
0     ../data/raw/train/Glioma/labels/Tr-gl_1184.txt      0  
1  ../data/raw/train/Meningioma/labels/Tr-me_0190...      1  
2     ../data/raw/train/Pituitary/labels/p (669).txt      3  
3  ../data/raw/train/Meningioma/labels/Tr-me_0501...      1  
4  ../data/raw/train/Pituitary/labels/Tr-pi_0777.txt      3  


In [10]:
print("First image:", train_df["image"].iloc[0])
print("Image exists:", Path(train_df["image"].iloc[0]).exists())

First image: ../data/raw/train/Glioma/images/Tr-gl_1184.jpg
Image exists: False


In [6]:
def load_image(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(image, IMG_SIZE)

    image.set_shape((224, 224, 3))

    return image, label

In [8]:
def create_dataset(table, shuffle=False):

    paths = table["image"].tolist()
    labels = table["label"].tolist()

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)

    ds = ds.map(load_image)

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [10]:
train_ds = create_dataset(train_df, shuffle=True)
val_ds = create_dataset(val_df)

print("Datasets ready!")

Datasets ready!


In [12]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomContrast(0.1)
])

In [13]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [14]:
resnet_base = tf.keras.applications.ResNet50V2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

resnet_base.trainable = False

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step


In [ ]:
## MobileNetV2

inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

x = tf.keras.layers.Rescaling(
    1./127.5,
    offset=-1
)(x)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    4,
    activation="softmax"
)(x)

model = tf.keras.Model(inputs, outputs)